# Name That LLM — 18 分类 LLM 文本溯源

给定一段 40–250 词的英文文本，判断它由 18 个候选大模型中的哪一个生成。比赛指标是 **balanced accuracy**（18 个类别 recall 的平均值）。

## 方案

| 步骤 | 内容 | 设备 |
|---|---|---|
| 0 | 配置、数据读取、按 `prompt_id` 分组 5 折 | CPU |
| 1 | 字符/词 TF-IDF + 排版风格特征 + 逻辑回归（强基线） | CPU |
| 2 | DeBERTa-v3-large 微调（fp16 + 梯度检查点 + 梯度累积），5 折 | GPU T4 15GB |
| 3 | 可选：用公开开放池 `crtproduction/name-that-llm-open-pool` 扩充训练数据 | GPU |
| 4 | 按 OOF logloss 搜索权重，加权几何平均融合 | CPU |
| 5 | 写 `submission.parquet` / `submission.csv` | CPU |

## 关键设计

- 训练集 150 个 prompt × 18 个模型；隐藏测试的 prompt 在训练集中**没有出现**。所以 CV 必须按 `prompt_id` 分组，否则同题泄漏、线下分数虚高。
- 隐藏测试每个 prompt 只有约 1–5 条回答，**不要**使用“同题 18 个模型一一分配”的后处理（这是 CV 陷阱）。
- T4 15GB：DeBERTa-v3-large 使用 fp16、动态 padding、梯度检查点、batch 4 × 累积 4；OOM 时自动降到 batch 2。
- 支持断点续跑：每折结果保存为 `{tag}_fold{k}.npz`，重跑自动跳过已完成折。
- 没有 GPU / 没有 transformer 模型时，自动退回 TF-IDF 提交，notebook 仍能跑通。

## Kaggle 运行步骤

1. 新建 Kaggle Notebook，右侧 Accelerator 选 **GPU T4 ×1**（15GB）。
2. 附加 DeBERTa-v3-large 模型（二选一）：
   - **Add Input → Models**：搜索 `microsoft/deberta-v3-large`，选择公开模型（例如 `romanchirkov1/microsoftdeberta-v3-large`）附加；
   - 或把 HuggingFace snapshot 上传为 Kaggle Dataset，再 Add Input。
3. （强烈推荐）**Add Input → Datasets**：搜索 `crtproduction/name-that-llm-open-pool` 并附加。该数据集为 CC BY-SA 4.0 公开外部数据，包含 190,588 条、覆盖全部 18 个类别。使用前请自行确认当季比赛规则允许公开外部数据。
4. 保持互联网关闭（Notebook 离线运行）。代码优先读取已附加的模型/数据集；找不到本地模型时会尝试 HuggingFace 名称，离线失败则自动跳过 Transformer。
5. 从头运行。产物：`/kaggle/working/submission.parquet`。本地运行时可用环境变量 `NTL_MODEL_PATH` 指定模型目录、`NTL_POOL_PATH` 指定开放池文件。

## 时间预算（T4 15GB，5 折，DeBERTa-v3-large，仅供参考）

| 设置 | 约耗时 |
|---|---|
| 只用 train.csv（无开放池） | 1–1.5 小时 |
| `POOL_PER_CLASS=1000`, `EPOCHS_POOL=2` | ~3 小时 |
| `POOL_PER_CLASS=1500`, `EPOCHS_POOL=2`（默认） | ~4.5–5 小时 |
| `POOL_PER_CLASS=2500`, `EPOCHS_POOL=2` | ~7–8 小时（接近 9h 上限） |

> 开放池默认每类采样 1,500 条（约 2.7 万条/折），用 `match_weight` 让排版/长度更接近真实 train。若时间充足，把 `POOL_PER_CLASS` 提到 2500，分数通常更好；也可以把 `EPOCHS_POOL` 降到 1 来控制时间。
>
> 建议第一次运行时先看 fold0 的耗时和显存：如果单折超过 ~1.5 小时，把 `POOL_PER_CLASS` 降到 1000（或设 0 只用 train.csv），避免 9 小时上限内跑不完。
>
> 注意：混入开放池后的 CV 会比 LB 高几个点（数据提供者原话），线下分数只用于选模型/融合权重，不要据此判断 LB。


In [ ]:
# ============================================================
# 0. 配置、路径、随机种子
# ============================================================
import os, re, gc, glob, json, math, time, random, contextlib, warnings
from pathlib import Path

# 必须在 import torch 之前设置：减少显存碎片
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
N_FOLDS = 5

# ---------- Transformer（T4 15GB 保守配置） ----------
MAX_LEN = 384                  # 训练文本最长 274 词，384 token 基本不截断
TRAIN_BS = 4                   # 单卡 batch
GRAD_ACC = 4                   # 梯度累积，等效 batch = 16
LR = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LABEL_SMOOTH = 0.05
GRAD_CKPT = True               # 显存不足时保持 True
USE_FP16 = True                # T4 不支持 bf16
EVAL_BS = 16
MAX_TRANSFORMERS = 1           # 有多个 backbone 时可设 2（时间约翻倍）
RUN_TRANSFORMER = "auto"       # "auto" / True / False
EPOCHS_NO_POOL = 4
EPOCHS_POOL = 2

# ---------- 公开开放池（可选，强烈推荐） ----------
POOL_PER_CLASS = 1500          # 每类最多采样条数；0 = 不用开放池
POOL_WEIGHT_COL = "match_weight"
POOL_MIN_WORDS, POOL_MAX_WORDS = 36, 275
EXCLUDE_KB_PROXY = False       # True = 去掉 Kaggle Benchmarks proxy 行（更严格，但少约 37% 数据）

# ---------- 断点续跑 ----------
FORCE_RETRAIN = False          # True = 忽略已保存的 fold 结果

# 本地 / Kaggle 模型目录；也可设环境变量 NTL_MODEL_PATH
MODEL_HINTS = [
    os.environ.get("NTL_MODEL_PATH"),
    "/kaggle/input/models/romanchirkov1/microsoftdeberta-v3-large/transformers/default/1",
    "/kaggle/input/deberta-v3-large",
    "/kaggle/input/deberta-v3-base",
    "/kaggle/input/name-that-llm-deberta/deberta-v3-large",
]
HF_FALLBACKS = ["microsoft/deberta-v3-large", "microsoft/deberta-v3-base"]

random.seed(SEED)
np.random.seed(SEED)


def find_data_dir():
    # 依次尝试：环境变量 -> Kaggle 比赛目录 -> 本地 data/ -> /kaggle/input 下含 train/test 的目录。
    cands = []
    env = os.environ.get("DATA_DIR")
    if env:
        cands.append(Path(env))
    cands += [
        Path("/kaggle/input/competitions/name-that-llm"),
        Path("/kaggle/input/name-that-llm"),
        Path("data"),
        Path("../data"),
        Path("input"),
    ]
    for p in cands:
        try:
            if (p / "train.csv").exists() and (p / "test.csv").exists():
                return p.resolve()
        except OSError:
            pass
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv"]:
        for f in sorted(glob.glob(pat)):
            p = Path(f).parent
            if (p / "test.csv").exists():
                return p.resolve()
    for f in sorted(Path.cwd().glob("**/train.csv")):
        if (f.parent / "test.csv").exists():
            return f.parent.resolve()
    raise FileNotFoundError(
        "找不到 train.csv/test.csv。请设置环境变量 DATA_DIR 指向比赛数据目录，"
        "或在 Kaggle Notebook 里 Add Input 比赛数据。"
    )


def find_out_dir():
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    out = Path(os.environ.get("OUT_DIR", "output"))
    out.mkdir(parents=True, exist_ok=True)
    return out.resolve()


DATA = find_data_dir()
OUT = find_out_dir()
print("DATA:", DATA)
print("OUT :", OUT)
print(f"SEED={SEED} | N_FOLDS={N_FOLDS} | MAX_LEN={MAX_LEN} | "
      f"bs={TRAIN_BS}x{GRAD_ACC} | pool/class={POOL_PER_CLASS}")


In [ ]:
# ============================================================
# 1. 数据、标签、按 prompt 分组的 5 折
# ============================================================
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder

tr = pd.read_csv(DATA / "train.csv", dtype={"id": str, "prompt_id": str})
te = pd.read_csv(DATA / "test.csv", dtype={"id": str})

need_tr = {"id", "prompt_id", "text", "model"}
need_te = {"id", "text"}
assert need_tr <= set(tr.columns), f"train.csv 缺列: {need_tr - set(tr.columns)}"
assert need_te <= set(te.columns), f"test.csv 缺列: {need_te - set(te.columns)}"

tr["text"] = tr["text"].fillna("").astype(str)
te["text"] = te["text"].fillna("").astype(str)
tr["model"] = tr["model"].astype(str).str.strip()
tr["id"] = tr["id"].astype(str)
te["id"] = te["id"].astype(str)

le = LabelEncoder().fit(tr["model"])
y = le.transform(tr["model"]).astype(np.int64)
K, N = len(le.classes_), len(tr)

# GroupKFold: 同一个 prompt 的 18 条回答只会出现在同一折
folds = list(GroupKFold(n_splits=N_FOLDS).split(tr, y, tr["prompt_id"]))
np.savez_compressed(
    OUT / "folds.npz",
    **{f"fold{k}": np.asarray(v) for k, (_, v) in enumerate(folds)},
    y=y,
)

print(f"train={tr.shape}  test={te.shape}  classes={K}  prompts={tr['prompt_id'].nunique()}")
print("每类样本数:", tr.groupby("model").size().describe()[["min", "max"]].to_dict())
print("每 prompt 条数:", tr.groupby("prompt_id").size().value_counts().to_dict())
lens = tr["text"].str.split().str.len()
print("词数分位:", {q: int(lens.quantile(q)) for q in [0, .5, .9, .99, 1]})
print("标签顺序:", list(le.classes_))


## 1. TF-IDF + 排版风格特征 + 逻辑回归（CPU 基线）

不同模型的标点习惯、破折号/引号/冒号、段落和列表格式、句长节奏差异很大。字符 n-gram 对排版极其敏感，再叠加一组手工风格统计量，得到一个便宜且稳的基线。

- 字符 TF-IDF：1–5 元，保留大小写和标点，300k 特征。
- 词 TF-IDF：1–2 元，标点也作为 token，200k 特征。
- 风格特征：长度、词长、句长、大小写/数字/空白比例、每千字符的各类标点、列表行比例。
- 分类器：`LogisticRegression(C=20, class_weight="balanced")`，与 balanced accuracy 对齐。

产出：OOF 概率（用于融合权重）和测试集概率（5 折模型平均）。


In [ ]:
# ============================================================
# 2. TF-IDF + 风格特征 + 逻辑回归（CPU 基线）
# ============================================================
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, log_loss

# 标点/排版是区分模型的重要信号；保留大小写
PUNCT = [",", ".", "?", "!", ";", ":", "—", "–", "-", "(", ")", "“", "”", "‘", "’",
         "'", '"', "…", "•", "#", "*", "`", "/", "%", "&"]


def style_features(texts):
    rows = []
    for t in texts:
        n = max(len(t), 1)
        ws = re.findall(r"\S+", t)
        nw = max(len(ws), 1)
        wl = np.array([len(w) for w in ws], dtype=np.float32) if ws else np.array([0.], np.float32)
        sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]
        sl = np.array([len(s.split()) for s in sents], dtype=np.float32) if sents else np.array([0.], np.float32)
        alpha = sum(c.isalpha() for c in t)
        upper = sum(c.isupper() for c in t)
        digits = sum(c.isdigit() for c in t)
        lines = t.split("\n")
        bullets = sum(bool(re.match(r"^\s*(?:[-*•]|\d+[.)])\s+", l)) for l in lines)
        row = [
            math.log1p(n), math.log1p(nw), wl.mean() / 10, wl.std() / 10,
            math.log1p(len(lines)), math.log1p(len(re.findall(r"\n\s*\n", t)) + 1),
            upper / max(alpha, 1), digits / n, sum(c.isspace() for c in t) / n,
            bullets / max(len(lines), 1), sl.mean() / 20, sl.std() / 20,
            len(sents) / nw,
        ]
        row += [t.count(p) * 1000 / n for p in PUNCT]
        rows.append(row)
    return np.asarray(rows, dtype=np.float32)


def make_tfidf():
    return FeatureUnion([
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(1, 5), lowercase=False,
                                 sublinear_tf=True, min_df=3, max_features=300_000)),
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=False,
                                 sublinear_tf=True, min_df=2, max_features=200_000,
                                 token_pattern=r"(?u)\b\w+\b|[^\w\s]")),
    ])


S_tr = style_features(tr["text"].tolist())
S_te = style_features(te["text"].tolist())

oof_lin = np.zeros((N, K), dtype=np.float32)
test_lin = np.zeros((len(te), K), dtype=np.float32)
lin_scores = []
t0 = time.time()

for k, (tri, vai) in enumerate(folds):
    vec = make_tfidf()
    Xa = vec.fit_transform(tr["text"].iloc[tri])
    Xb = vec.transform(tr["text"].iloc[vai])
    Xt = vec.transform(te["text"])

    scaler = StandardScaler()
    Xa = hstack([Xa, csr_matrix(scaler.fit_transform(S_tr[tri]))]).tocsr()
    Xb = hstack([Xb, csr_matrix(scaler.transform(S_tr[vai]))]).tocsr()
    Xt = hstack([Xt, csr_matrix(scaler.transform(S_te))]).tocsr()

    clf = LogisticRegression(C=20, max_iter=500, class_weight="balanced")
    clf.fit(Xa, y[tri])

    oof_lin[vai] = clf.predict_proba(Xb)
    test_lin += clf.predict_proba(Xt) / N_FOLDS
    ba = balanced_accuracy_score(y[vai], oof_lin[vai].argmax(1))
    lin_scores.append(ba)
    print(f"fold {k}: BA={ba:.4f}  ({time.time()-t0:.0f}s)", flush=True)

lin_ba = balanced_accuracy_score(y, oof_lin.argmax(1))
lin_ll = log_loss(y, oof_lin, labels=np.arange(K))
print(f"TF-IDF + style  OOF BA={lin_ba:.4f} ± {np.std(lin_scores):.4f}  logloss={lin_ll:.4f}")

np.savez_compressed(
    OUT / "linear_oof.npz",
    oof=oof_lin.astype(np.float32),
    test=test_lin.astype(np.float32),
    test_ids=te["id"].to_numpy(dtype=str),
    classes=le.classes_.astype(str),
    max_len=0,
)


## 2. 可选：加载公开开放池

`crtproduction/name-that-llm-open-pool` 汇总了 55 个公开数据集，共 **190,588** 条、覆盖全部 18 个类别，且已做去重、去泄漏、去低质过滤。核心用法：

- 每个类别最多采样 `POOL_PER_CLASS` 条（默认 1,500），用 `match_weight` 作为抽样权重，让排版和长度更接近真实 train。
- 每折用不同随机种子采样一次开放池，相当于数据增强。
- 如果当前环境没有附加该数据集，自动跳过，只用 train.csv 训练。

> 开放池的 `match_weight` 是在全部 train prompt 上拟合的，所以混入后的 CV 会比 LB 高几个点；这是正常现象。


In [ ]:
# ============================================================
# 3. 可选：加载公开开放池
# ============================================================
def find_pool_file():
    env = os.environ.get("NTL_POOL_PATH")
    cands = [Path(env)] if env else []
    cands += [
        Path("/kaggle/input/name-that-llm-open-pool/ntl_open_pool.parquet"),
        Path("data/ntl_open_pool.parquet"),
        Path("../input/name-that-llm-open-pool/ntl_open_pool.parquet"),
    ]
    for p in cands:
        try:
            if p.exists():
                return p.resolve()
        except OSError:
            pass
    for pat in ["/kaggle/input/*/ntl_open_pool.parquet",
                "/kaggle/input/*/*/ntl_open_pool.parquet",
                "/kaggle/input/*/*/*/ntl_open_pool.parquet"]:
        hits = sorted(glob.glob(pat))
        if hits:
            return Path(hits[0]).resolve()
    return None


def sample_pool(pool, per_class, seed):
    # 每个类别按 match_weight 抽样 per_class 条；match_weight 的说明见数据集主页。
    rng = np.random.default_rng(seed)
    parts = []
    for c in le.classes_:
        d = pool[pool["model"] == c]
        n = min(int(per_class), len(d))
        if n <= 0:
            continue
        if n >= len(d):
            take = d
        else:
            w = pd.to_numeric(d[POOL_WEIGHT_COL], errors="coerce").fillna(1.0).to_numpy(dtype=np.float64)
            w = np.clip(w, 0, None)
            if not np.isfinite(w).all() or w.sum() <= 0:
                w = np.ones(len(d), dtype=np.float64)
            idx = rng.choice(len(d), size=n, replace=False, p=w / w.sum())
            take = d.iloc[idx]
        parts.append(take)
    if not parts:
        return pool.iloc[[]].copy()
    out = pd.concat(parts, ignore_index=True)
    return out.sample(frac=1.0, random_state=int(rng.integers(1 << 31))).reset_index(drop=True)


POOL = None
pool_path = find_pool_file()
if pool_path is not None and POOL_PER_CLASS > 0:
    print("开放池:", pool_path)
    cols_want = ["text", "model", "n_words", POOL_WEIGHT_COL,
                 "flag_near_train_prompt", "flag_train_prompt_regen", "flag_kb_proxy"]
    try:
        pool = pd.read_parquet(pool_path, columns=cols_want)
    except Exception:
        # 旧版本或列名不同：读全表再取需要的列
        try:
            pool = pd.read_parquet(pool_path)
        except Exception as e:
            print("读取开放池失败，忽略:", repr(e)[:200])
            pool = None
        if pool is not None:
            for c in cols_want:
                if c not in pool.columns:
                    pool[c] = np.nan
            pool = pool[cols_want]
    if pool is not None:
        pool["text"] = pool["text"].fillna("").astype(str)
        pool["model"] = pool["model"].astype(str).str.strip()
        pool = pool[pool["model"].isin(le.classes_) & (pool["text"].str.len() > 0)]
        pool["n_words"] = pd.to_numeric(pool["n_words"], errors="coerce")
        pool = pool[pool["n_words"].between(POOL_MIN_WORDS, POOL_MAX_WORDS) | pool["n_words"].isna()]
        for flag in ["flag_near_train_prompt", "flag_train_prompt_regen"]:
            pool = pool[~pool[flag].fillna(False).astype(bool)]
        if EXCLUDE_KB_PROXY:
            pool = pool[~pool["flag_kb_proxy"].fillna(False).astype(bool)]
        pool[POOL_WEIGHT_COL] = pd.to_numeric(pool[POOL_WEIGHT_COL], errors="coerce").fillna(1.0).clip(lower=1e-3)
        pool = pool.drop_duplicates(subset=["text", "model"]).reset_index(drop=True)
        if len(pool):
            POOL = pool
            print(f"开放池可用: {len(POOL):,} 条 | 类别数 {POOL['model'].nunique()} | "
                  f"每类中位数 {int(POOL.groupby('model').size().median())}")
            print(POOL.groupby("model").size().sort_values().to_string())
        else:
            print("开放池过滤后为空，忽略。")
else:
    print("未使用开放池。可附加 crtproduction/name-that-llm-open-pool 来提升分数。")


## 3. DeBERTa-v3-large 微调（GPU）

- 模型：`microsoft/deberta-v3-large`，18 分类头。
- 混合精度 fp16（T4 不支持 bf16），动态 padding（每批补到该批最长），余弦学习率 + 10% warmup，梯度裁剪 1.0，label smoothing 0.05。
- 显存：梯度检查点 + batch 4 × 累积 4（等效 batch 16）；若 OOM 自动降到 batch 2 × 累积 8。
- 每折在验证集上按 **logloss 最低** 保存最佳 epoch 的权重，再用它预测测试集。
- 每折保存 `{tag}_fold{k}.npz`，包含验证集概率、测试集概率、验证集索引、test id、标签顺序，重跑会自动校验并跳过。
- 模型路径自动搜索：`/kaggle/input/**/config.json` 中优先 `deberta-v3-large`，其次 `deberta-v3-base`、`roberta-*`；找不到本地模型时尝试 HuggingFace 名称（仅在有网络时有效）。

> 如果你只附加了 base 模型，代码会自动用它；分数会低一些，但流程不变。


In [ ]:
# ============================================================
# 4. Transformer 工具：设备、模型发现、分词、训练/推理
# ============================================================
try:
    import torch
    import torch.nn.functional as F
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_cosine_schedule_with_warmup
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print("torch/transformers 不可用，跳过 Transformer:", repr(e)[:200])

if HAS_TORCH:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    DEVICE = "cpu"

HAS_GPU = bool(HAS_TORCH and DEVICE == "cuda")
DO_TRANSFORMER = HAS_GPU if RUN_TRANSFORMER == "auto" else bool(RUN_TRANSFORMER)
if DO_TRANSFORMER and not HAS_GPU:
    print("RUN_TRANSFORMER=True 但没有可用 GPU，已关闭 Transformer。")
    DO_TRANSFORMER = False

if HAS_GPU:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM {props.total_memory/1e9:.1f} GB | torch {torch.__version__}")
else:
    print("GPU: 无")
print("DO_TRANSFORMER:", DO_TRANSFORMER)


def _read_config(d):
    try:
        with open(Path(d) / "config.json", "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}


def model_tag(path):
    s = str(path).lower()
    for kw in ["deberta-v3-large", "deberta-v3-base", "deberta-v3-small",
               "roberta-large", "roberta-base"]:
        if kw in s:
            return kw
    if "deberta" in s and "large" in s:
        return "deberta-large"
    if "deberta" in s and "base" in s:
        return "deberta-base"
    cfg = _read_config(path)
    name = str(cfg.get("_name_or_path") or Path(path).name)
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", name)[:48] or "model"


def _has_tokenizer(d):
    d = Path(d)
    return any((d / f).exists() for f in
               ["tokenizer.json", "tokenizer_config.json", "vocab.json",
                "spm.model", "sentencepiece.bpe.model"])


def find_model_dirs():
    # 用户显式给出的路径优先保留，不强制 tokenizer 文件齐全
    direct = []
    for p in MODEL_HINTS:
        if p and (Path(p) / "config.json").exists():
            direct.append(Path(p))

    found = []
    pats = ["/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
            "/kaggle/input/*/*/*/config.json", "/kaggle/input/*/*/*/*/config.json",
            "/kaggle/input/*/*/*/*/*/config.json",
            "/kaggle/input/*/*/*/*/*/*/config.json"]
    for pat in pats:
        for f in glob.glob(pat):
            d = Path(f).parent
            if (d / "config.json").exists() and _has_tokenizer(d):
                found.append(d)

    # 去重保序
    dirs = [Path(d) for d in dict.fromkeys(str(d) for d in (direct + found))]

    def score(d):
        s = str(d).lower()
        for kw, sc in [("deberta-v3-large", 100), ("deberta-v3-base", 90),
                       ("roberta-large", 80), ("roberta-base", 70),
                       ("deberta", 60), ("bert", 10)]:
            if kw in s:
                return sc
        return 0

    dirs = sorted(dirs, key=lambda d: (score(d), str(d)), reverse=True)
    return dirs


def candidate_models():
    out, seen = [], set()
    for d in find_model_dirs():
        tag = model_tag(d)
        if tag not in seen:
            seen.add(tag)
            out.append((tag, str(d)))
    if not out:
        for name in HF_FALLBACKS:
            out.append((model_tag(name), name))
    return out


if HAS_TORCH:
    def tokenize_texts(texts, tok, max_len):
        return [tok(t, truncation=True, max_length=max_len, add_special_tokens=True)["input_ids"]
                for t in texts]

    def make_batches(ids, indices, bs, shuffle, rng):
        # 按长度分桶，减少 padding；不改变样本集合。
        idx = np.asarray(indices, dtype=np.int64)
        if len(idx) == 0:
            return []
        if shuffle:
            idx = rng.permutation(idx)
            chunk = max(bs * 32, 256)
            parts = []
            for i in range(0, len(idx), chunk):
                part = idx[i:i + chunk]
                order = np.argsort([len(ids[j]) for j in part], kind="stable")
                parts.append(part[order])
            idx = np.concatenate(parts)
        else:
            idx = idx[np.argsort([len(ids[j]) for j in idx], kind="stable")]
        return [idx[i:i + bs] for i in range(0, len(idx), bs)]

    def collate(ids, batch, pad_id):
        L = max(len(ids[j]) for j in batch)
        x = np.full((len(batch), L), pad_id, dtype=np.int64)
        m = np.zeros((len(batch), L), dtype=np.int64)
        for r, j in enumerate(batch):
            l = len(ids[j])
            x[r, :l] = ids[j]
            m[r, :l] = 1
        return torch.from_numpy(x).to(DEVICE), torch.from_numpy(m).to(DEVICE)

    def amp_ctx():
        if USE_FP16 and DEVICE == "cuda":
            try:
                return torch.autocast(device_type="cuda", dtype=torch.float16)
            except Exception:
                return torch.cuda.amp.autocast(dtype=torch.float16)
        return contextlib.nullcontext()

    def make_scaler():
        enabled = bool(USE_FP16 and DEVICE == "cuda")
        try:
            return torch.amp.GradScaler("cuda", enabled=enabled)
        except Exception:
            return torch.cuda.amp.GradScaler(enabled=enabled)

    @torch.no_grad()
    def predict_probs(model, ids, indices, pad_id, bs=EVAL_BS):
        model.eval()
        indices = np.asarray(indices, dtype=np.int64)
        out = np.zeros((len(indices), K), dtype=np.float32)
        if len(indices) == 0:
            return out
        order = np.argsort([len(ids[j]) for j in indices], kind="stable")
        cur_bs = bs
        pos = 0
        while pos < len(order):
            sel = order[pos:pos + cur_bs]
            batch = [int(indices[i]) for i in sel]
            try:
                x, m = collate(ids, batch, pad_id)
                with amp_ctx():
                    logits = model(input_ids=x, attention_mask=m).logits
                out[sel] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
                pos += len(sel)
            except RuntimeError as e:
                if "out of memory" not in str(e).lower() or cur_bs <= 1:
                    raise
                cur_bs = max(1, cur_bs // 2)
                print(f"  推理显存不足，eval batch 降为 {cur_bs}", flush=True)
                gc.collect()
                if HAS_GPU:
                    torch.cuda.empty_cache()
        return out

    def train_fold(model_path, tag, k, train_idx, val_idx, ids_all, y_all,
                   ids_test, test_ids, pad_id, epochs, bs=TRAIN_BS, acc=GRAD_ACC):
        t0 = time.time()
        torch.manual_seed(SEED * 100 + k)
        np.random.seed(SEED * 100 + k)
        random.seed(SEED * 100 + k)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, num_labels=K, ignore_mismatched_sizes=True,
            local_files_only=Path(model_path).exists())
        if GRAD_CKPT and hasattr(model, "gradient_checkpointing_enable"):
            try:
                model.gradient_checkpointing_enable()
            except Exception as e:
                print("gradient checkpointing 不可用，继续:", repr(e)[:120])
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False
        model.to(DEVICE)

        no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias",
                    "layer_norm.weight", "layer_norm.bias")
        params = [
            {"params": [p for n, p in model.named_parameters()
                        if p.requires_grad and not any(t in n for t in no_decay)],
             "weight_decay": WEIGHT_DECAY},
            {"params": [p for n, p in model.named_parameters()
                        if p.requires_grad and any(t in n for t in no_decay)],
             "weight_decay": 0.0},
        ]
        opt = torch.optim.AdamW(params, lr=LR)
        rng = np.random.default_rng(SEED * 1000 + k)
        batches_per_epoch = math.ceil(len(train_idx) / bs)
        steps_per_epoch = math.ceil(batches_per_epoch / acc)
        total_steps = max(1, steps_per_epoch * epochs)
        warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
        sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
        scaler = make_scaler()
        y_t = torch.as_tensor(y_all, dtype=torch.long, device=DEVICE)

        print(f"fold{k}: train={len(train_idx):,} val={len(val_idx):,} "
              f"micro_bs={bs} accum={acc} epochs={epochs} steps={total_steps}", flush=True)

        best_ll, best_ba, best_state, best_ep = float("inf"), 0.0, None, -1
        for ep in range(epochs):
            model.train()
            batches = make_batches(ids_all, train_idx, bs, True, rng)
            opt.zero_grad(set_to_none=True)
            for step, batch in enumerate(batches):
                x, m = collate(ids_all, batch, pad_id)
                yb = y_t[torch.as_tensor(batch, dtype=torch.long, device=DEVICE)]
                with amp_ctx():
                    logits = model(input_ids=x, attention_mask=m).logits
                    loss = F.cross_entropy(logits, yb, label_smoothing=LABEL_SMOOTH)
                scaler.scale(loss / acc).backward()
                if (step + 1) % acc == 0 or (step + 1) == len(batches):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                    sched.step()
                    opt.zero_grad(set_to_none=True)
                if (step + 1) % 200 == 0:
                    print(f"  fold{k} ep{ep+1} {step+1}/{len(batches)} "
                          f"loss={float(loss.detach()):.4f} ({time.time()-t0:.0f}s)", flush=True)

            val_p = predict_probs(model, ids_all, val_idx, pad_id)
            if not np.isfinite(val_p).all():
                print(f"fold{k} ep{ep+1}: val 概率出现 NaN/Inf，跳过该 epoch", flush=True)
                continue
            val_ll = float(log_loss(y_all[val_idx], np.clip(val_p, 1e-9, 1.0), labels=np.arange(K)))
            val_ba = float(balanced_accuracy_score(y_all[val_idx], val_p.argmax(1)))
            print(f"fold{k} ep{ep+1}: val_logloss={val_ll:.4f} val_BA={val_ba:.4f} "
                  f"({time.time()-t0:.0f}s)", flush=True)
            if val_ll < best_ll - 1e-5:
                best_ll, best_ba, best_ep = val_ll, val_ba, ep + 1
                best_state = {n: v.detach().cpu().clone() for n, v in model.state_dict().items()}

        if best_state is not None:
            model.load_state_dict(best_state)
        model.to(DEVICE)
        val_p = predict_probs(model, ids_all, val_idx, pad_id)
        test_p = predict_probs(model, ids_test, np.arange(len(ids_test)), pad_id)

        path = OUT / f"{tag}_fold{k}.npz"
        np.savez_compressed(
            path,
            val=val_p.astype(np.float32),
            test=test_p.astype(np.float32),
            idx=np.asarray(val_idx, dtype=np.int64),
            test_ids=np.asarray(test_ids, dtype=str),
            classes=np.asarray(le.classes_, dtype=str),
            best_ll=np.float64(best_ll),
            best_ba=np.float64(best_ba),
            best_epoch=np.int64(best_ep),
            max_len=np.int64(MAX_LEN),
            epochs=np.int64(epochs),
            pool_per_class=np.int64(POOL_PER_CLASS if POOL is not None else 0),
        )
        del model, best_state, opt
        gc.collect()
        if HAS_GPU:
            torch.cuda.empty_cache()
        print(f"fold{k} done: BA={best_ba:.4f} best_ep={best_ep} 保存 {path.name} "
              f"({time.time()-t0:.0f}s)", flush=True)
        return val_p, test_p, best_ll, best_ba

    def train_fold_safe(model_path, tag, k, train_idx, val_idx, ids_all, y_all,
                        ids_test, test_ids, pad_id, epochs):
        attempts = [(TRAIN_BS, GRAD_ACC)]
        if TRAIN_BS > 1:
            new_bs = max(1, TRAIN_BS // 2)
            new_acc = max(1, int(round(GRAD_ACC * TRAIN_BS / new_bs)))
            attempts.append((new_bs, new_acc))
        last_err = None
        for i, (bs, acc) in enumerate(attempts):
            try:
                return train_fold(model_path, tag, k, train_idx, val_idx, ids_all, y_all,
                                  ids_test, test_ids, pad_id, epochs, bs=bs, acc=acc)
            except RuntimeError as e:
                last_err = e
                if "out of memory" not in str(e).lower() or i == len(attempts) - 1:
                    raise
                print(f"  显存不足，降为 batch={attempts[i+1][0]} × 累积={attempts[i+1][1]} 重试 ...",
                      flush=True)
                gc.collect()
                if HAS_GPU:
                    torch.cuda.empty_cache()
                time.sleep(2)
        raise last_err

    def fold_file(tag, k):
        return OUT / f"{tag}_fold{k}.npz"

    def try_load_fold(tag, k, val_idx, test_ids):
        p = fold_file(tag, k)
        if FORCE_RETRAIN or not p.exists():
            return None
        try:
            z = np.load(p)
            if int(z["max_len"]) != MAX_LEN:
                print(f"  {p.name}: MAX_LEN 不匹配，重训")
                return None
            expected_pool = POOL_PER_CLASS if POOL is not None else 0
            if "pool_per_class" not in z or int(z["pool_per_class"]) != expected_pool:
                print(f"  {p.name}: 开放池配置不匹配，重训")
                return None
            if not np.array_equal(z["idx"], np.asarray(val_idx)):
                print(f"  {p.name}: 验证集索引不匹配，重训")
                return None
            if not np.array_equal(z["test_ids"].astype(str), np.asarray(test_ids, dtype=str)):
                print(f"  {p.name}: test id 不匹配，重训")
                return None
            if not np.array_equal(z["classes"].astype(str), le.classes_.astype(str)):
                print(f"  {p.name}: 标签顺序不匹配，重训")
                return None
            return z["val"], z["test"], float(z["best_ll"]), float(z["best_ba"])
        except Exception as e:
            print(f"  {p.name}: 读取失败，重训 ({repr(e)[:120]})")
            return None


In [ ]:
# ============================================================
# 5. 运行 Transformer 5 折（可断点续跑）
# ============================================================
oof_tf, test_tf, tf_tags = {}, {}, []

if not DO_TRANSFORMER:
    print("跳过 Transformer（无 GPU / torch 不可用 / RUN_TRANSFORMER=False）。")
else:
    cands = candidate_models()
    print("候选模型:", cands)
    done_models = 0

    for tag, model_path in cands:
        if done_models >= MAX_TRANSFORMERS:
            break
        print("\n" + "=" * 70)
        print(f"模型 {tag}: {model_path}")
        try:
            tok = AutoTokenizer.from_pretrained(model_path, local_files_only=Path(model_path).exists())
            if tok.pad_token_id is None:
                tok.pad_token = tok.eos_token
            pad_id = tok.pad_token_id
            if pad_id is None:
                raise ValueError("tokenizer 没有 pad_token_id")
            print("分词 train/test ...", flush=True)
            ids_tr = tokenize_texts(tr["text"].tolist(), tok, MAX_LEN)
            ids_te = tokenize_texts(te["text"].tolist(), tok, MAX_LEN)
            tl = np.array([len(x) for x in ids_tr])
            print(f"token 长度: median={int(np.median(tl))} p95={int(np.percentile(tl,95))} "
                  f"max={int(tl.max())} | 截断比例={(tl >= MAX_LEN).mean():.1%}")
        except Exception as e:
            print("加载 tokenizer/分词失败，跳过该模型:", repr(e)[:300])
            continue

        epochs = EPOCHS_POOL if POOL is not None else EPOCHS_NO_POOL
        oof = np.zeros((N, K), dtype=np.float32)
        test = np.zeros((len(te), K), dtype=np.float32)
        folds_ok = 0

        for k, (tri, vai) in enumerate(folds):
            cached = try_load_fold(tag, k, vai, te["id"].to_numpy(dtype=str))
            if cached is not None:
                vp, tp, ll, ba = cached
                print(f"fold{k}: 读取缓存 BA={ba:.4f} logloss={ll:.4f}")
            else:
                # 本折的开放池采样（不同折用不同随机种子，相当于数据增强）
                if POOL is not None:
                    sub = sample_pool(POOL, POOL_PER_CLASS, seed=SEED + k)
                    ids_pool = tokenize_texts(sub["text"].tolist(), tok, MAX_LEN)
                    y_pool = le.transform(sub["model"].astype(str)).astype(np.int64)
                    print(f"fold{k}: 使用开放池 {len(sub):,} 条 (pool/class={POOL_PER_CLASS})",
                          flush=True)
                else:
                    ids_pool, y_pool = [], np.array([], dtype=np.int64)

                ids_all = ids_tr + ids_pool
                y_all = np.concatenate([y, y_pool]).astype(np.int64)
                train_idx = np.concatenate([
                    np.asarray(tri, dtype=np.int64),
                    np.arange(N, N + len(ids_pool), dtype=np.int64),
                ])
                try:
                    vp, tp, ll, ba = train_fold_safe(
                        model_path, tag, k, train_idx, vai, ids_all, y_all,
                        ids_te, te["id"].to_numpy(dtype=str), pad_id, epochs,
                    )
                except Exception as e:
                    print(f"fold{k} 训练失败: {repr(e)[:300]}")
                    break

            if not (np.isfinite(vp).all() and np.isfinite(tp).all()):
                print(f"fold{k} 预测含 NaN/Inf，跳过该模型")
                break
            oof[vai] = vp
            test += tp / N_FOLDS
            folds_ok += 1

        if folds_ok == N_FOLDS:
            oof_tf[tag], test_tf[tag] = oof, test
            tf_tags.append(tag)
            done_models += 1
            print(f"模型 {tag} 完成 5 折: OOF BA="
                  f"{balanced_accuracy_score(y, oof.argmax(1)):.4f}, "
                  f"logloss={log_loss(y, oof, labels=np.arange(K)):.4f}")
        else:
            print(f"模型 {tag} 只完成 {folds_ok}/{N_FOLDS} 折，跳过融合。")


## 4. 融合与提交

- 读取 TF-IDF 的 OOF/测试概率，以及磁盘上所有 **5 折齐全** 的 Transformer 结果。
- 在 OOF 上最小化 logloss，搜索各模型的权重，然后做加权几何平均：`softmax(Σ w_i · log p_i)`。
- 打印每个模型的 OOF balanced accuracy、融合后的 BA/logloss，以及 18 个类别的 per-class recall（balancing 指标下要重点看这个）。
- 融合后按 argmax 生成 `submission.parquet`，并做 id/标签/空值校验。

> 没有做“同题 18 个模型一一分配”的后处理：隐藏测试每个 prompt 只有约 1–5 条回答，训练集里的 18 条/题结构在测试集不存在，强行分配会掉分（公开讨论中已验证）。


In [ ]:
# ============================================================
# 6. 融合 + 生成提交
# ============================================================
from scipy.optimize import minimize
from sklearn.metrics import balanced_accuracy_score, log_loss, confusion_matrix


def logloss(p, yy):
    return float(-np.log(np.clip(p[np.arange(len(yy)), yy], 1e-9, None)).mean())


def geometric_blend(probs, weights):
    z = sum(w * np.log(np.clip(p, 1e-9, None)) for w, p in zip(weights, probs))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


oofs, tests = {}, {}

# TF-IDF
lin_path = OUT / "linear_oof.npz"
if lin_path.exists():
    z = np.load(lin_path)
    if not np.array_equal(z["test_ids"].astype(str), te["id"].to_numpy(dtype=str)):
        raise ValueError("linear_oof.npz 的 test id 与当前 test.csv 不一致，请删除后重跑 TF-IDF 单元。")
    if "classes" in z and not np.array_equal(z["classes"].astype(str), le.classes_.astype(str)):
        raise ValueError("linear_oof.npz 的标签顺序与当前 train.csv 不一致，请删除后重跑 TF-IDF 单元。")
    if z["oof"].shape != (N, K) or z["test"].shape != (len(te), K):
        raise ValueError("linear_oof.npz 的形状与当前数据不一致，请删除后重跑 TF-IDF 单元。")
    oofs["tfidf_lr"] = z["oof"].astype(np.float32)
    tests["tfidf_lr"] = z["test"].astype(np.float32)

# Transformer 各折结果
tags = sorted({re.sub(r"_fold\d+\.npz$", "", p.name) for p in OUT.glob("*_fold*.npz")})
for tag in tags:
    files = [OUT / f"{tag}_fold{k}.npz" for k in range(N_FOLDS)]
    if not all(p.exists() for p in files):
        print(f"跳过 {tag}: 缺折")
        continue
    o = np.zeros((N, K), dtype=np.float32)
    t = np.zeros((len(te), K), dtype=np.float32)
    ok = True
    for k, p in enumerate(files):
        try:
            zz = np.load(p)
            if int(zz["max_len"]) != MAX_LEN:
                print(f"跳过 {tag}: MAX_LEN 不一致")
                ok = False
                break
            if not np.array_equal(zz["idx"], np.asarray(folds[k][1])):
                print(f"跳过 {tag}: fold{k} 索引不一致")
                ok = False
                break
            if not np.array_equal(zz["test_ids"].astype(str), te["id"].to_numpy(dtype=str)):
                print(f"跳过 {tag}: fold{k} test id 不一致")
                ok = False
                break
            if "classes" in zz and not np.array_equal(zz["classes"].astype(str), le.classes_.astype(str)):
                print(f"跳过 {tag}: fold{k} 标签顺序不一致")
                ok = False
                break
            o[zz["idx"]] = zz["val"]
            t += zz["test"] / N_FOLDS
        except Exception as e:
            print(f"跳过 {tag}: fold{k} 读取/校验失败 ({repr(e)[:120]})")
            ok = False
            break
    if ok:
        if not (np.isfinite(o).all() and np.isfinite(t).all()):
            print(f"跳过 {tag}: 预测含 NaN/Inf")
            continue
        oofs[tag], tests[tag] = o, t

if not oofs:
    raise RuntimeError("没有可用的 OOF 结果。")

for n in oofs:
    print(f"{n:24s} OOF BA={balanced_accuracy_score(y, oofs[n].argmax(1)):.4f}  "
          f"logloss={logloss(oofs[n], y):.4f}")


def fit_weights(oof_dict):
    ns = list(oof_dict)
    if len(ns) == 1:
        return {ns[0]: 1.0}

    def objective(a):
        w = np.exp(a - a.max())
        w = w / w.sum()
        p = geometric_blend([oof_dict[n] for n in ns], w)
        return logloss(p, y)

    res = minimize(objective, np.zeros(len(ns)), method="Nelder-Mead",
                   options={"maxiter": 2000, "xatol": 1e-4, "fatol": 1e-7})
    w = np.exp(res.x - res.x.max())
    w = w / w.sum()
    return dict(zip(ns, w))


weights = fit_weights(oofs)
names = list(oofs)
print("融合权重:", {n: round(float(weights[n]), 3) for n in names})

oof_blend = geometric_blend([oofs[n] for n in names], [weights[n] for n in names])
test_blend = geometric_blend([tests[n] for n in names], [weights[n] for n in names])
print(f"融合 OOF: BA={balanced_accuracy_score(y, oof_blend.argmax(1)):.4f}  "
      f"logloss={logloss(oof_blend, y):.4f}")

# 每类 recall 诊断（balanced accuracy = 这些值的平均）
cm = confusion_matrix(y, oof_blend.argmax(1), labels=np.arange(K))
recall = cm.diagonal() / np.maximum(cm.sum(1), 1)
print("每类 recall:")
for c, r in sorted(zip(le.classes_, recall), key=lambda x: x[1]):
    print(f"  {c:26s} {r:.3f}")

# ---------- 提交 ----------
pred = le.inverse_transform(test_blend.argmax(1))

sample = None
for sp in [DATA / "sample_submission.parquet", DATA / "sample_submission.csv"]:
    if sp.exists():
        sample = pd.read_parquet(sp) if sp.suffix == ".parquet" else pd.read_csv(sp)
        break

sample_ok = False
if sample is not None and len(sample) == len(te):
    id_col = "id" if "id" in sample.columns else sample.columns[0]
    if set(sample[id_col].astype(str)) == set(te["id"]):
        sub = sample[[id_col]].copy()
        sub = sub.rename(columns={id_col: "id"})
        sub["id"] = sub["id"].astype(str)
        sub["model"] = sub["id"].map(dict(zip(te["id"], pred)))
        sample_ok = True
if not sample_ok:
    sub = pd.DataFrame({"id": te["id"].to_numpy(dtype=str), "model": pred})

assert len(sub) == len(te), "提交行数与 test.csv 不一致"
assert sub["model"].notna().all(), "存在空预测"
assert sub["id"].is_unique, "id 有重复"
assert set(sub["model"]) <= set(le.classes_), "预测标签不在合法集合内"

sub.to_parquet(OUT / "submission.parquet", index=False)
sub.to_csv(OUT / "submission.csv", index=False)
np.save(OUT / "test_probs.npy", test_blend.astype(np.float32))

print(f"\n已保存: {OUT/'submission.parquet'}  shape={sub.shape}")
print(sub["model"].value_counts().to_string())
sub.head(10)


## 后续可提升的方向

1. **增加 backbone**：把 `MAX_TRANSFORMERS` 设为 2，同时融合 DeBERTa-v3-large 和 base；或换不同随机种子的同模型，做 seed averaging。
2. **扩大开放池**：`POOL_PER_CLASS` 从 1500 提到 2500（时间约 7–8 小时），通常还能涨点。
3. **外部生成数据**：比赛允许公开外部数据；可用候选模型对开放池的 prompt 补生成缺失的 prompt×label 组合（数据集里的 `panel_gaps.csv` 列出了缺口），但要注意规则和标签可靠性。
4. **Surprisal / 困惑度特征**：用一个小的离线语言模型计算 token log-prob 统计，再和 TF-IDF/风格特征拼给线性模型。
5. **分层建模**：先预测厂商（OpenAI/Google/Anthropic/Qwen/ZAI/Moonshot），再预测厂商内模型；对 Claude/Gemini 这类同门模型可能有帮助。
6. **类别偏差校准**：在 OOF 上搜索每个类别的 logit bias，使 balanced accuracy 最大；务必用嵌套 group CV 验证增益，避免过拟合（本 notebook 默认不做）。

## 环境与依赖

- Python 3.10+，`numpy`、`pandas`、`scikit-learn`、`scipy`、`pyarrow`。
- Transformer 部分需要 `torch`、`transformers`、`sentencepiece`（Kaggle GPU 镜像自带）。
- 没有 GPU 时 notebook 仍可跑通，只用 TF-IDF 生成提交。
